# Tiny Dreamer Highway — Colab Git + Drive Setup

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook follows the project rule: GitHub is the source of truth for code, and Google Drive stores artifacts such as checkpoints, plots, and videos.

## Runtime flow

1. Mount Google Drive.
2. Clone or pull the latest repository snapshot from GitHub.
3. Install the package and dependencies.
4. Run a small sanity check before training.
5. Save outputs into the Drive-backed `artifacts/` folder.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
WORKTREE = Path('/content/CSC_580_Final_Project')

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'checkpoints', ARTIFACT_ROOT / 'media', ARTIFACT_ROOT / 'logs']:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

print('Artifact root:', ARTIFACT_ROOT)

print('Repository URL:', REPO_URL)


In [ ]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi

In [ ]:
%%bash
set -e
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip
python -m pip install -e .


## Dependency source of truth



This notebook installs the project directly from the repository with `python -m pip install -e .`. The dependency versions come from `pyproject.toml`, so the notebook does not duplicate the full package list.



The project now tracks a Colab-friendly scientific stack built around `numpy` 2.x and current compatible releases of `scipy`, `gymnasium`, and `highway-env`.



If Colab replaces a preloaded package during installation, use **Runtime -> Restart session** once and rerun the notebook from the top before testing the environment import.


In [ ]:
from importlib.metadata import version

print('numpy version:', version('numpy'))
print('scipy version:', version('scipy'))
print('gymnasium version:', version('gymnasium'))
print('highway-env version:', version('highway-env'))


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.cli import summarize_config

config_path = PROJECT_ROOT / 'examples' / 'base_experiment.yaml'
config = load_experiment_config(config_path)
print(summarize_config(config))

In [ ]:
from tiny_dreamer_highway.envs.highway_factory import make_highway_env

env = make_highway_env(config.env)
observation, info = env.reset()
action = env.action_space.sample()
next_observation, reward, terminated, truncated, info = env.step(action)
env.close()

print('Observation shape:', getattr(observation, 'shape', None))
print('Action shape:', getattr(action, 'shape', None))
print('Reward:', reward)
print('Episode ended:', bool(terminated or truncated))

## Next step

After this sanity pass succeeds, add cells for replay warm-start collection, world model smoke tests, and training artifacts written to `ARTIFACT_ROOT`.